# E791 $D^+\to\pi^-\pi^+\pi^+$ — coefficient closure with adaptive normalization

This notebook mirrors the validated regular-grid closure test in `02_fit_dynamic_parameters.ipynb`, changing only the normalization quadrature.

Setup:

1. E791 Fit-2 coefficients define the injected truth;
2. all resonance masses, widths, spins and radii are fixed;
3. $\rho(770)$ is fixed to $1+0i$;
4. all other coefficients float as unbounded Cartesian $(x,y)$ parameters;
5. one pseudo-data sample is generated;
6. one broad randomized starting point is drawn independently of fit bounds;
7. the JAX gradient is checked against central finite differences;
8. exactly one Minuit fit is performed with `tolerance=1e-4`.

The adaptive grid is built once from the fixed dynamical basis functions and then frozen throughout the coefficient-only fit.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    AdaptiveDalitzGrid,
    DecayChannel,
    DecayModel,
    Minimizer,
    NonResonant,
    Parameter,
    RealImag,
    Resonance,
    enable_x64,
    weighted_resample,
)

enable_x64()


## 1. E791 Fit-2 injected coefficients


In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))

fit2_polar = {
    "sigma":   (1.17, 205.7),
    "rho770":  (1.00,   0.0),
    "NR":      (0.48,  57.3),
    "f0_980":  (0.43, 165.0),
    "f2_1270": (0.76,  57.3),
    "f0_1370": (0.26, 105.4),
    "rho1450": (0.14, 319.1),
}

def polar_to_xy(r, phase_deg):
    phase = np.deg2rad(phase_deg)
    return r * np.cos(phase), r * np.sin(phase)

def internal_xy(name):
    r, phase = fit2_polar[name]
    if name == "NR":
        phase += 180.0
    return polar_to_xy(r, phase)

truth_xy = {name: internal_xy(name) for name in fit2_polar}

print(f"{'component':10s} {'x truth':>12s} {'y truth':>12s}")
for name, (x, y) in truth_xy.items():
    print(f"{name:10s} {x:12.6f} {y:12.6f}")


## 2. Build the coefficient-only model

The floating Cartesian coefficients are deliberately **unbounded**, exactly as in notebook 02.


In [ ]:
truth = {}

def free_coefficient(name):
    x_truth, y_truth = truth_xy[name]
    truth[f"{name}.x"] = float(x_truth)
    truth[f"{name}.y"] = float(y_truth)
    return RealImag(
        Parameter.coefficient(f"{name}.x", 0.0, owner=name, step=0.01),
        Parameter.coefficient(f"{name}.y", 0.0, owner=name, step=0.01),
    )

coefficients = {
    "sigma": free_coefficient("sigma"),
    "rho770": RealImag(1.0, 0.0),
    "NR": free_coefficient("NR"),
    "f0_980": free_coefficient("f0_980"),
    "f2_1270": free_coefficient("f2_1270"),
    "f0_1370": free_coefficient("f0_1370"),
    "rho1450": free_coefficient("rho1450"),
}

components = [
    Resonance("sigma",   (0,1), coefficients["sigma"],   mass=0.4780, width=0.3240, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho770",  (0,1), coefficients["rho770"],  mass=0.7693, width=0.1502, spin=1, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_980",  (0,1), coefficients["f0_980"],  mass=0.9750, width=0.0440, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f2_1270", (0,1), coefficients["f2_1270"], mass=1.2750, width=0.1850, spin=2, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_1370", (0,1), coefficients["f0_1370"], mass=1.4340, width=0.1730, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho1450", (0,1), coefficients["rho1450"], mass=1.4650, width=0.3100, spin=1, resonance_radius=3.0, parent_radius=3.0),
    NonResonant(coefficients["NR"]),
]

model = DecayModel(channel, components)

print("Free fit parameters:")
for p in model.parameters:
    if not p.fixed:
        print(f"  {p.name:16s} bounds={p.bounds}")
print("number of free parameters =", sum(not p.fixed for p in model.parameters))


## 3. Build the adaptive normalization grid

Every non-constant fixed dynamical basis function is used as a refinement probe. Coefficients do not enter the probes, so the grid is independent of the fit point and can be reused during minimization.


In [ ]:
built_components = model.amplitude_model.components

probes = tuple(
    (lambda data, dynamics=component.function: dynamics(data, None))
    for component in built_components
    if component.name != "NR"
)
probe_names = [component.name for component in built_components if component.name != "NR"]

BASE_RESOLUTION = 48
MAX_DEPTH = 5
ADAPTIVE_TOLERANCE = 0.08

adaptive_builder = AdaptiveDalitzGrid(
    channel.parent_mass,
    channel.daughter_masses,
    base_resolution=BASE_RESOLUTION,
    max_depth=MAX_DEPTH,
    tolerance=ADAPTIVE_TOLERANCE,
    max_cells=2_000_000,
)
adaptive = adaptive_builder.build(probes)
norm = adaptive.sample

print("probes             :", probe_names)
print(f"base cells          : {BASE_RESOLUTION**2:,}")
print(f"adaptive leaf cells : {adaptive.size:,}")
print("max depth reached   :", int(jnp.max(adaptive.depth)))
print("weight min/max      :", float(jnp.min(norm.weights)), float(jnp.max(norm.weights)))
print("sum leaf uv area    :", float(jnp.sum(adaptive.du * adaptive.dv)))


## 4. Visualize the adaptive grid


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(
    np.asarray(norm.s12), np.asarray(norm.s13),
    c=np.asarray(adaptive.depth), s=1.5, rasterized=True,
)
fig.colorbar(sc, ax=ax, label="refinement depth")
ax.set_xlabel(r"$s_{12}$ [GeV$^2$]")
ax.set_ylabel(r"$s_{13}$ [GeV$^2$]")
ax.set_title("Adaptive normalization grid for the E791 model")
plt.show()


## 5. Quadrature sanity checks


In [ ]:
integral_one = float(jnp.mean(norm.weights * jnp.ones_like(norm.weights)))
print("integral of 1      =", integral_one)
print("finite weights    =", bool(jnp.all(jnp.isfinite(norm.weights))))
print("sum leaf uv area =", float(jnp.sum(adaptive.du * adaptive.dv)))
assert abs(float(jnp.sum(adaptive.du * adaptive.dv)) - 1.0) < 1e-10


## 6. Generate the same pseudo-data configuration as notebook 02


In [ ]:
N_POOL = 1_000_000
N_DATA = 100_000

pool = model.generate_phase_space(N_POOL, seed=2000)
truth_cache_pool = model.prepare_cache(pool, norm)
truth_intensity, truth_normalization = truth_cache_pool.evaluate(truth)
target_weights = pool.weights * truth_intensity

data = weighted_resample(
    jax.random.key(791), pool, target_weights, N_DATA, replace=True
)

print(f"candidate pool      : {pool.size:,}")
print(f"pseudo-data events : {data.size:,}")
print("truth normalization:", float(truth_normalization))


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6.5))
h = ax.hist2d(np.asarray(data.s12), np.asarray(data.s13), bins=110)
fig.colorbar(h[3], ax=ax, label="events")
ax.set_xlabel(r"$s_{12}$ [GeV$^2$]")
ax.set_ylabel(r"$s_{13}$ [GeV$^2$]")
ax.set_title(r"E791 Fit-2 pseudo-data: $D^+\to\pi^-\pi^+\pi^+$")
plt.show()


## 7. Likelihood and broad randomized start

As in notebook 02, the interval `(-2.5, 2.5)` is only the random-start distribution. It is **not** a Minuit limit.


In [ ]:
cache = model.prepare_cache(data, norm)

def nll(values):
    intensity, normalization = cache.evaluate(values)
    return (
        -jnp.sum(jnp.log(jnp.clip(intensity, min=1e-300)))
        + data.size * jnp.log(normalization)
    )

minimizer = Minimizer(
    nll,
    model.parameters,
    tolerance=1e-4,
    verbose=2,
)

START_SEED = 314159
START_RANGE = (-2.5, 2.5)
rng = np.random.default_rng(START_SEED)
start_values = {
    p.name: float(rng.uniform(*START_RANGE))
    for p in model.parameters
    if not p.fixed
}

print(f"Random start seed   = {START_SEED}")
print(f"Start draw interval = {START_RANGE} (NOT a fit limit)")
print(f"{'parameter':16s} {'truth':>11s} {'start':>11s} {'delta':>11s}")
for p in model.parameters:
    if p.fixed:
        continue
    t = truth[p.name]
    s = start_values[p.name]
    print(f"{p.name:16s} {t:11.6f} {s:11.6f} {s-t:+11.6f}")

print(f"NLL(truth) = {float(nll(truth)):.6f}")
print(f"NLL(start) = {float(nll(start_values)):.6f}")


## 8. Check the JAX gradient before minimization


In [ ]:
gradient_check = minimizer.check_gradient(
    start_values,
    step_scale=1e-5,
    print_table=True,
)


## 9. Perform exactly one fit


In [ ]:
result = minimizer.fit(
    start_values=start_values,
    simplex=False,
    ncall=100000,
)

fit_values = {
    p.name: float(result.values[p.name])
    for p in model.parameters
    if not p.fixed
}

print("valid            :", bool(result.valid))
print("NLL(start)       :", float(nll(start_values)))
print("NLL(truth)       :", float(nll(truth)))
print("NLL(fit)         :", float(result.fval))
print("NLL(fit)-truth   :", float(result.fval - nll(truth)))
print("EDM              :", float(result.fmin.edm))
print("function calls   :", int(result.nfcn))


## 10. Closure table


In [ ]:
rows = []
print(f"{'parameter':16s} {'truth':>10s} {'start':>10s} {'fit':>10s} {'error':>10s} {'pull':>9s}")
for p in model.parameters:
    if p.fixed:
        continue
    t = float(truth[p.name])
    s = float(start_values[p.name])
    f = float(result.values[p.name])
    e = float(result.errors[p.name])
    pull = (f - t) / e
    rows.append((p.name, t, s, f, e, pull))
    print(f"{p.name:16s} {t:10.5f} {s:10.5f} {f:10.5f} {e:10.5f} {pull:9.3f}")


## 11. Truth vs start vs fit


In [ ]:
names = [p.name for p in model.parameters if not p.fixed]
x = np.arange(len(names))
truth_arr = np.array([truth[n] for n in names])
start_arr = np.array([start_values[n] for n in names])
fit_arr = np.array([result.values[n] for n in names], dtype=float)
fit_err = np.array([result.errors[n] for n in names], dtype=float)

fig, ax = plt.subplots(figsize=(13, 5.5))
ax.scatter(x, truth_arr, marker="x", s=70, label="truth")
ax.scatter(x, start_arr, marker="o", s=30, label="random start")
ax.errorbar(x, fit_arr, yerr=fit_err, fmt=".", capsize=2, label="fit")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=60, ha="right")
ax.set_ylabel("Cartesian coefficient")
ax.set_title("Coefficient closure with adaptive normalization")
ax.legend()
fig.tight_layout()
plt.show()


## 12. Projection: start, fit and truth


In [ ]:
projection_cache = model.prepare_cache(pool, norm)

def projection(values, bins):
    intensity, _ = projection_cache.evaluate(values)
    w = np.asarray(pool.weights * intensity)
    h12, _ = np.histogram(np.asarray(pool.s12), bins=bins, weights=w)
    h13, _ = np.histogram(np.asarray(pool.s13), bins=bins, weights=w)
    return h12 + h13

sdata = np.concatenate([np.asarray(data.s12), np.asarray(data.s13)])
bins = np.linspace(sdata.min(), sdata.max(), 110)
centres = 0.5 * (bins[:-1] + bins[1:])
hd, _ = np.histogram(sdata, bins=bins)
hs = projection(start_values, bins)
ht = projection(truth, bins)
hf = projection(fit_values, bins)
for h in (hs, ht, hf):
    h *= hd.sum() / h.sum()

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.errorbar(centres, hd, yerr=np.sqrt(np.maximum(hd, 1)), fmt=".", label="toy")
ax.step(centres, hs, where="mid", label="random start")
ax.step(centres, hf, where="mid", label="fit")
ax.step(centres, ht, where="mid", linestyle="--", label="truth")
ax.set_xlabel(r"$m^2(\pi^-\pi^+)$ [GeV$^2$]")
ax.set_ylabel("entries / bin")
ax.legend()
plt.show()


## Interpretation

This notebook should reproduce the coefficient closure behavior of notebook 02 while using many fewer normalization points. The key comparison is `NLL(fit)-NLL(truth)`, the pull distribution, and consistency of the fitted coefficients. If closure differs materially from the regular grid, the adaptive quadrature itself — not the minimizer configuration — is the first thing to investigate.
